In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType
from delta.tables import DeltaTable

# Source data
SOURCE = "s3a://data5035-spring26/drone_data.json"

# Output database — tables will be written here
OUTPUT_DB = "default"

# Table names
DIM_DETECTOR_TABLE  = f"{OUTPUT_DB}.dim_detector"
DIM_GPS_TABLE       = f"{OUTPUT_DB}.dim_gps_unit"
FACT_TABLE          = f"{OUTPUT_DB}.fact_radiation_readings"

print("Configuration loaded.")

In [0]:
# Read the raw denormalized JSON from S3
source_df = spark.read.json(SOURCE)

print(f"Row count: {source_df.count()}")
source_df.printSchema()
display(source_df.limit(5))

In [0]:
def build_scd2(df, natural_keys, tracked_cols, timestamp_col):
    """
    Build a SCD Type 2 dimension from a deduplicated DataFrame.

    Parameters
    ----------
    df            : DataFrame of distinct dimension records (natural key + tracked attributes)
    natural_keys  : list of column names that uniquely identify the entity (e.g. ['unit_number', 'detector_type'])
    tracked_cols  : list of column names whose changes trigger a new version (e.g. ['calibration_precision'])
    timestamp_col : column name used to order versions chronologically (e.g. 'calibration_timestamp')

    Returns
    -------
    DataFrame with SCD Type 2 columns added:
        effective_start  — when this version became active
        effective_end    — when this version was superseded (null = still active)
        is_current       — boolean flag for the active version
    """

    # Deduplicate: one row per unique combination of natural key + tracked attributes
    version_df = df.select(natural_keys + tracked_cols + [timestamp_col]).dropDuplicates()

    # Order versions per entity by the calibration timestamp
    window = Window.partitionBy(natural_keys).orderBy(timestamp_col)

    scd2_df = (
        version_df
        .withColumn("effective_start", F.col(timestamp_col).cast("timestamp"))
        .withColumn(
            "effective_end",
            F.lead(timestamp_col).over(window).cast("timestamp")
        )
        .withColumn(
            "is_current",
            F.col("effective_end").isNull()
        )
        .withColumn(
            "surrogate_key",
            F.md5(F.concat_ws("|", *[F.col(c) for c in natural_keys], F.col("effective_start").cast(StringType())))
        )
        # Backdate the first version of each entity to the beginning of time.
        # Without this, readings taken before the first calibration timestamp
        # produce a null FK in the fact table because no effective window covers them.
        .withColumn("row_num", F.row_number().over(window))
        .withColumn(
            "effective_start",
            F.when(F.col("row_num") == 1, F.lit("1900-01-01").cast("timestamp"))
             .otherwise(F.col("effective_start"))
        )
        .drop("row_num")
    )

    return scd2_df


print("SCD Type 2 helper function defined.")

In [0]:
# --- Step 1: Extract each detector type as a uniform DataFrame ---

cesium_df = source_df.select(
    F.lit("CESIUM_137").alias("detector_type"),
    F.col("CESIUM_137_DETECTOR_UNIT_NUMBER").alias("unit_number"),
    F.col("CESIUM_137_DETECTOR_CALIBRATION_PRECISION").alias("calibration_precision"),
    F.col("CESIUM_137_DETECTOR_CALIBRATION_TIMESTAMP").alias("calibration_timestamp")
)

gamma_df = source_df.select(
    F.lit("GAMMA").alias("detector_type"),
    F.col("GAMMA_DETECTOR_UNIT_NUMBER").alias("unit_number"),
    F.col("GAMMA_DETECTOR_CALIBRATION_PRECISION").alias("calibration_precision"),
    F.col("GAMMA_DETECTOR_CALIBRATION_TIMESTAMP").alias("calibration_timestamp")
)

thorium_df = source_df.select(
    F.lit("THORIUM_232").alias("detector_type"),
    F.col("THORIUM_232_DETECTOR_UNIT_NUMBER").alias("unit_number"),
    F.col("THORIUM_232_DETECTOR_CALIBRATION_PRECISION").alias("calibration_precision"),
    F.col("THORIUM_232_DETECTOR_CALIBRATION_TIMESTAMP").alias("calibration_timestamp")
)

# --- Step 2: Union all detector types into one DataFrame ---
all_detectors_df = cesium_df.unionByName(gamma_df).unionByName(thorium_df)

# --- Step 3: Apply SCD Type 2 logic ---
dim_detector_df = build_scd2(
    df            = all_detectors_df,
    natural_keys  = ["unit_number", "detector_type"],
    tracked_cols  = ["calibration_precision"],
    timestamp_col = "calibration_timestamp"
)

# --- Step 4: Write to Delta table ---
(
    dim_detector_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_DETECTOR_TABLE)
)

print(f"dim_detector written with {dim_detector_df.count()} rows.")
display(spark.table(DIM_DETECTOR_TABLE))

In [0]:
# --- Step 1: Extract GPS unit attributes ---
gps_df = source_df.select(
    F.col("GPS_UNIT_NUMBER").alias("unit_number"),
    F.col("GPS_UNIT_CALIBRATION_PRECISION").alias("calibration_precision"),
    F.col("GPS_UNIT_CALIBRATION_TIMESTAMP").alias("calibration_timestamp")
)

# --- Step 2: Apply SCD Type 2 logic ---
dim_gps_df = build_scd2(
    df            = gps_df,
    natural_keys  = ["unit_number"],
    tracked_cols  = ["calibration_precision"],
    timestamp_col = "calibration_timestamp"
)

# --- Step 3: Write to Delta table ---
(
    dim_gps_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_GPS_TABLE)
)

print(f"dim_gps_unit written with {dim_gps_df.count()} rows.")
display(spark.table(DIM_GPS_TABLE))

In [0]:
# Load the current dimension tables
dim_detector  = spark.table(DIM_DETECTOR_TABLE)
dim_gps       = spark.table(DIM_GPS_TABLE)

# Cast collection timestamp once for join comparisons
readings_df = source_df.withColumn(
    "collection_ts", F.col("COLLECTION_TIMESTAMP").cast("timestamp")
)

# --- Helper: join to dim_detector for a specific detector type ---
# Matches on unit_number + detector_type + event falls within effective window
def join_detector(df, unit_col, detector_type, key_alias):
    detector_version = dim_detector.filter(F.col("detector_type") == detector_type)
    return df.join(
        detector_version.select(
            F.col("unit_number"),
            F.col("surrogate_key").alias(key_alias),
            F.col("effective_start"),
            F.col("effective_end")
        ),
        on=(
            (F.col(unit_col) == detector_version["unit_number"]) &
            (F.col("collection_ts") >= F.col("effective_start")) &
            (
                F.col("effective_end").isNull() |
                (F.col("collection_ts") < F.col("effective_end"))
            )
        ),
        how="left"
    ).drop("unit_number", "effective_start", "effective_end")


# --- Join each detector dimension ---
fact_df = readings_df
fact_df = join_detector(fact_df, "CESIUM_137_DETECTOR_UNIT_NUMBER",  "CESIUM_137",  "dim_cesium_detector_key")
fact_df = join_detector(fact_df, "GAMMA_DETECTOR_UNIT_NUMBER",       "GAMMA",       "dim_gamma_detector_key")
fact_df = join_detector(fact_df, "THORIUM_232_DETECTOR_UNIT_NUMBER", "THORIUM_232", "dim_thorium_detector_key")

# --- Join GPS unit dimension ---
fact_df = fact_df.join(
    dim_gps.select(
        F.col("unit_number"),
        F.col("surrogate_key").alias("dim_gps_unit_key"),
        F.col("effective_start"),
        F.col("effective_end")
    ),
    on=(
        (F.col("GPS_UNIT_NUMBER") == dim_gps["unit_number"]) &
        (F.col("collection_ts") >= F.col("effective_start")) &
        (
            F.col("effective_end").isNull() |
            (F.col("collection_ts") < F.col("effective_end"))
        )
    ),
    how="left"
).drop("unit_number", "effective_start", "effective_end")

# --- Select final fact table columns ---
fact_df = fact_df.select(
    # Surrogate key for the fact row itself
    F.md5(F.col("collection_ts").cast(StringType())).alias("reading_id"),
    # Measurements
    F.col("collection_ts").alias("collection_timestamp"),
    F.col("CESIUM_137_LEVEL").alias("cesium_137_level"),
    F.col("GAMMA_LEVEL").alias("gamma_level"),
    F.col("THORIUM_232_LEVEL").alias("thorium_232_level"),
    F.col("GPS_LAT").alias("gps_lat"),
    F.col("GPS_LNG").alias("gps_lng"),
    # Foreign keys to dimensions
    F.col("dim_cesium_detector_key"),
    F.col("dim_gamma_detector_key"),
    F.col("dim_thorium_detector_key"),
    F.col("dim_gps_unit_key")
)

# --- Write to Delta table ---
(
    fact_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FACT_TABLE)
)

print(f"fact_radiation_readings written with {fact_df.count()} rows.")
display(spark.table(FACT_TABLE))

In [0]:
# ── Row count summary ────────────────────────────────────────────────────────
# Confirmation all three tables loaded with data.
print("=== Table Row Counts ===")
print(f"dim_detector:             {spark.table(DIM_DETECTOR_TABLE).count()} rows")
print(f"dim_gps_unit:             {spark.table(DIM_GPS_TABLE).count()} rows")
print(f"fact_radiation_readings:  {spark.table(FACT_TABLE).count()} rows")

# ── Referential integrity check ──────────────────────────────────────────────
# Every FK column in the fact table must resolve to a dimension record.
# A non-zero count here means orphaned rows that will break joins downstream.
print("\n=== Null FK Check (all should be 0) ===")
fact = spark.table(FACT_TABLE)
for fk in ["dim_cesium_detector_key", "dim_gamma_detector_key", "dim_thorium_detector_key", "dim_gps_unit_key"]:
    nulls = fact.filter(F.col(fk).isNull()).count()
    print(f"  {fk}: {nulls} nulls")

# ── Dimension table spot-checks ───────────────────────────────────────────────
# Show only the active/current SCD rows for detectors so stale history
# doesn't clutter the output.
print("\n=== dim_detector: Current Versions ===")
display(spark.table(DIM_DETECTOR_TABLE).filter(F.col("is_current") == True))

# GPS units are displayed in full — useful for auditing all historical
# location assignments, not just the latest active record.
print("=== dim_gps_unit: All Versions ===")
display(spark.table(DIM_GPS_TABLE))